# Joining Tables

Until now each query read from a single table. But the real use of a
relational database is connecting tables. Which member borrowed which book?
That answer lives across three tables: loan, member, and book. **JOIN**
combines rows from two or more tables based on a related column.

The diagram below compares all four joins on a small example of member and loan.
Carol has no loan, so she appears only when the left table is kept (LEFT and
FULL). The last loan points to a member id that is not in the member table, so
it appears only when the right table is kept (RIGHT and FULL).

<img src="images/library_joins.svg" alt="Comparison of inner, left, right and full joins on member and loan" width="680">

This is a small illustrative example. In your real Library a foreign key prevents
a loan from pointing to a missing member, so an outer join on the real tables
usually adds NULL on only one side, as the artist and album examples below show.

## INNER JOIN: matching rows

An INNER JOIN returns only rows that have a match in both tables:

```sql
SELECT columns
FROM table_a
JOIN table_b ON table_a.key = table_b.key;
```

In Chinook, each album has an artist_id. To see album titles together with
their artist name, we join album and artist:

In [ ]:
from rel_db_learning import get_chinook_connection, get_project_connection, run_sql
from rel_db_learning.widgets.self_assessment import sql_query_check

chinook = get_chinook_connection()
project = get_project_connection()

run_sql("""
    SELECT album.title, artist.name
    FROM album
    JOIN artist ON album.artist_id = artist.artist_id
    ORDER BY artist.name
    LIMIT 15
""", chinook)

The `ON` clause says how the tables connect: album.artist_id matches
artist.artist_id. Only albums with a matching artist appear (and the other way).

You can use table aliases to write less. Here `al` and `ar` are short names
for album and artist:

In [ ]:
run_sql("""
    SELECT al.title, ar.name
    FROM album al
    JOIN artist ar ON al.artist_id = ar.artist_id
    WHERE ar.name = 'Metallica'
""", chinook)

## Joining three tables

You can chain joins to connect more than two tables. In your Library, a loan
links a member to a book. To see who borrowed what, join loan to member and
to book:

In [ ]:
run_sql("""
    SELECT m.full_name, b.title
    FROM loan l
    JOIN member m ON l.member_id = m.member_id
    JOIN book b ON l.book_id = b.book_id
    ORDER BY m.full_name, b.title
    LIMIT 15
""", project)

## LEFT JOIN: keeping all rows from one side

An INNER JOIN drops rows with no match. The joins that keep unmatched rows
instead are called outer joins, and there are three of them: LEFT, RIGHT, and
FULL. We start with LEFT.

A LEFT JOIN keeps every row from the left table, filling the right side with NULL
where there is no match. In Chinook, some artists have no albums. An INNER JOIN
would hide them. A LEFT JOIN keeps them, with NULL in the album columns:

In [ ]:
run_sql("""
    SELECT ar.name, al.title
    FROM artist ar
    LEFT JOIN album al ON ar.artist_id = al.artist_id
    ORDER BY ar.name
    LIMIT 20
""", chinook)

To find only the artists with no albums, keep the rows where the right side
is NULL:

In [ ]:
run_sql("""
    SELECT ar.name
    FROM artist ar
    LEFT JOIN album al ON ar.artist_id = al.artist_id
    WHERE al.album_id IS NULL
    ORDER BY ar.name
    LIMIT 20
""", chinook)

## RIGHT JOIN: keeping all rows from the other side

A RIGHT JOIN is the mirror image of a LEFT JOIN: it keeps every row from the
right table. Any LEFT JOIN can be rewritten as a RIGHT JOIN by swapping the table
order, so in practice LEFT JOIN is the more common of the two.

Here is the same question as before (artists with no album), but written with a
RIGHT JOIN. Now album is on the left and artist is on the right, and we keep every
artist. The WHERE clause keeps only the artists whose album side came back NULL:

In [ ]:
run_sql("""
    SELECT ar.name
    FROM album al
    RIGHT JOIN artist ar ON al.artist_id = ar.artist_id
    WHERE al.album_id IS NULL
    ORDER BY ar.name
    LIMIT 20
""", chinook)

## FULL JOIN: keeping all rows from both sides

A FULL JOIN keeps every row from both tables, filling NULL wherever there is no
match. It is the union of what a LEFT JOIN and a RIGHT JOIN would keep.

The WHERE clause below keeps only the unmatched rows from either side. Note a
detail of this data: every album points to a real artist (a foreign key
guarantees it), so there are no albums without an artist. The only unmatched rows
here are artists without an album. The diagram at the top of this notebook shows
the more general case, where both sides can have unmatched rows.

In [ ]:
run_sql("""
    SELECT ar.name, al.title
    FROM artist ar
    FULL JOIN album al ON ar.artist_id = al.artist_id
    WHERE ar.artist_id IS NULL OR al.album_id IS NULL
    ORDER BY ar.name
    LIMIT 20
""", chinook)

## Self-join: a table joined to itself

Sometimes a table refers to itself. In Chinook, each employee has a
`reports_to` column pointing to another employee (their manager). To pair
each employee with their manager, we join employee to itself, using two
aliases:

In [ ]:
run_sql("""
    SELECT e.first_name || ' ' || e.last_name AS employee,
           m.first_name || ' ' || m.last_name AS manager
    FROM employee e
    LEFT JOIN employee m ON e.reports_to = m.employee_id
    ORDER BY employee
""", chinook)

The same table appears twice with different aliases (e for employee, m for
manager). The LEFT JOIN keeps the top employee, whose manager is NULL.

### Try it: compare the four join types

You have seen INNER, LEFT, RIGHT, and FULL joins on Chinook. The widget below runs all four on a small pair of tables, so you can switch between them and watch the result change.

Notice as you switch:

- LEFT keeps Carla, who has no loan, and fills the loan columns with NULL.
- RIGHT keeps the orphan loan, whose member is missing, and fills the member columns with NULL.
- FULL keeps both unmatched rows at once.

The generated SQL under each result shows the exact statement you would write.

In [ ]:
from rel_db_learning.widgets import join_visualizer

join_visualizer()

## Joins with aggregates

Joins combine naturally with GROUP BY. To count how many times each book was
borrowed, join book to loan and group:

In [ ]:
run_sql("""
    SELECT b.title, COUNT(l.loan_id) AS times_loaned
    FROM book b
    JOIN loan l ON b.book_id = l.book_id
    GROUP BY b.title
    ORDER BY times_loaned DESC, b.title
    LIMIT 10
""", project)

## Exercises

**Exercise 1 (INNER JOIN, three tables):** Return the `title` of every book
borrowed by the member named 'Dimitris Stavros'. Join loan, book, and member.

In [ ]:
my_sql_1 = "???"

sql_query_check(
    student_sql=my_sql_1,
    correct_sql="m04_nb_07_exercise_1",
    connection=project,
)

**Exercise 2 (JOIN + GROUP BY + HAVING):** Return each book `title` together
with how many times it was loaned, but only for books loaned more than once.
Join book and loan, group by title, and use HAVING.

In [ ]:
my_sql_2 = "???"

sql_query_check(
    student_sql=my_sql_2,
    correct_sql="m04_nb_07_exercise_2",
    connection=project,
)

**Exercise 3 (JOIN + GROUP BY + HAVING):** Return the `full_name` of every
member who has exactly 3 loans, together with that count. Join member and
loan, group by name, and use HAVING.

In [ ]:
my_sql_3 = "???"

sql_query_check(
    student_sql=my_sql_3,
    correct_sql="m04_nb_07_exercise_3",
    connection=project,
)

## The same in Python with Ibis

Ibis expresses joins too. Here is an inner join on Chinook, pairing albums
with their artist.

In [ ]:
import ibis
from rel_db_learning import get_ibis_chinook

ibis_chinook = get_ibis_chinook()
album = ibis_chinook.table("album")
artist = ibis_chinook.table("artist")

result = album.join(artist, album.artist_id == artist.artist_id).select(
    album.title, artist.name
).limit(10)
result.execute()

And the SQL Ibis generated:

In [ ]:
print(ibis.to_sql(result))

### Optional: try it on your Library

Connect to your Library with Ibis and try joining loan to book. This is
optional and not checked.

In [ ]:
from rel_db_learning import get_ibis_project

ibis_project = get_ibis_project()
loan = ibis_project.table("loan")
book = ibis_project.table("book")

loan.join(book, loan.book_id == book.book_id).select(book.title).limit(10).execute()

## Experiment freely

Try joining different tables. In your Library, connect loan to member, or
member_card to member. In Chinook, join track to album to artist for a full
picture.

In [ ]:
run_sql("""
    SELECT m.full_name, b.title, l.loan_date
    FROM loan l
    JOIN member m ON l.member_id = m.member_id
    JOIN book b ON l.book_id = b.book_id
    WHERE m.country = 'Greece'
    ORDER BY m.full_name, l.loan_date
""", project)

## What's next

You can now combine data across tables, the heart of relational querying.
Next you will learn to change data with UPDATE and DELETE, and you will grow
your Library with a new column.